Exploration notebook. The cited evaluation report is produced by
`python -m src.evaluate`; see DEVELOPMENT.md.

In [75]:

import torch
import os, sys
from torchvision.io import read_image

sys.path.append("..")
from src import preprocessing, nn_model
from src.preprocessing import check_data_input, check_model_input



In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
from torchvision import datasets
import PIL
from PIL import features

encoding_metadata = {
    "pillow_version": PIL.__version__,
    "jpeg_codec_version": features.version_codec("jpg"),
    "image_mode": "L",
    "jpeg_quality": 75,
    "png_save_options": "Pillow defaults",
}

# Export the paired evaluation inputs before reading individual files.
test_data = datasets.MNIST(root='../data', train=False, download=True)
os.makedirs('../data/test', exist_ok=True)
for source_id, (image, label) in enumerate(test_data):
    image.save(
        f'../data/test/{source_id}_label_{label}.jpg',
        format='JPEG', quality=75,
    )
    image.save(
        f'../data/test/{source_id}_label_{label}.png', format='PNG'
    )


In [ ]:
from pathlib import Path

checkpoint_path = Path("../models/checkpoint.pt")
loaded_checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=True
)
loaded_model = nn_model.NeuralNetwork().to(device)
loaded_model.load_state_dict(loaded_checkpoint['model_state_dict'])
loaded_model.eval()

In [ ]:
saved_preprocessing = loaded_checkpoint['preprocessing']

In [ ]:
path = "../data/test"

In [ ]:
images = []
sample_count = 10000
ce_loss = torch.nn.CrossEntropyLoss()

import torch.nn.functional as F
def brier_score_multiclass(preds, targets, num_classes = 10):
    """
    preds: Tensor of predicted class probabilities shape (N, K)
    targets: Tensor of class indices shape (N,)
    num_classes: Integer K representing total number of classes
    """
    # Convert integer targets to one-hot encoding
    targets_one_hot = F.one_hot(targets, num_classes=num_classes).float()
    
    # # Sum squared errors across classes, then average across samples.
    return torch.mean(torch.sum((preds - targets_one_hot) ** 2, dim=1))

from datetime import datetime
# Get current local date and time
current_time = datetime.now()
print(current_time.strftime("%Y-%m-%d %H:%M:%S"))

for i in range(sample_count):
    label = test_data.targets[i].item()
    jpg_path = os.path.join(path, f"{i}_label_{label}.jpg") 
    png_path = os.path.join(path, f"{i}_label_{label}.png")
    jpg = read_image(jpg_path)
    png = read_image(png_path)
    check_data_input(jpg)
    check_data_input(png)
    assert torch.equal(png, test_data.data[i].unsqueeze(0)), (
        f"PNG pixels differ from source {i}"
    )
    padding_jpg = preprocessing.padding_image(jpg)
    padding_png = preprocessing.padding_image(png)
    normalize_jpg = preprocessing.normalize_image(padding_jpg, saved_preprocessing['train_mean'], saved_preprocessing['train_std'])
    normalize_png = preprocessing.normalize_image(padding_png, saved_preprocessing['train_mean'], saved_preprocessing['train_std'])
    normalize_jpg = normalize_jpg.to(device).unsqueeze(0)
    normalize_png = normalize_png.to(device).unsqueeze(0)
    check_model_input(normalize_jpg)
    check_model_input(normalize_png)
    target = torch.tensor([label], dtype=torch.long, device=device)

    with torch.no_grad():
        jpg_logits = loaded_model(normalize_jpg)
        png_logits = loaded_model(normalize_png)
        jpg_pred = jpg_logits.argmax(dim=1).item()
        png_pred = png_logits.argmax(dim=1).item()
        jpg_celoss = ce_loss(jpg_logits, target).item()
        png_celoss = ce_loss(png_logits, target).item()

        jpg_prob = torch.softmax(jpg_logits, dim=-1)
        png_prob = torch.softmax(png_logits, dim=-1)
        jpg_brier = brier_score_multiclass(jpg_prob, target).item()
        png_brier = brier_score_multiclass(png_prob, target).item()
    images.append({
        "index": i,
        "label": label,
        "jpg_path": jpg_path,
        "png_path": png_path,
        "jpg_logits": jpg_logits,
        "png_logits": png_logits,
        "jpg_pred": jpg_pred,
        "png_pred": png_pred,
        'jpg_ce_loss': jpg_celoss,
        'png_ce_loss': png_celoss,
        'jpg_brier_loss': jpg_brier,
        'png_brier_loss': png_brier,
    })

current_time = datetime.now()
print(current_time.strftime("%Y-%m-%d %H:%M:%S"))


In [ ]:
import pandas as pd
images_df = pd.DataFrame(images)
assert len(images_df) == sample_count
assert images_df["index"].is_unique
assert set(images_df["index"]) == set(range(sample_count))

images_df["jpg_correct"] = images_df["jpg_pred"] == images_df["label"]
images_df["png_correct"] = images_df["png_pred"] == images_df["label"]
images_df["disagreement"] = images_df["png_pred"] != images_df["jpg_pred"]

# Direction: PNG → JPEG
images_df["correct_to_wrong"] = (
    images_df["png_correct"] & ~images_df["jpg_correct"]
)
images_df["wrong_to_correct"] = (
    ~images_df["png_correct"] & images_df["jpg_correct"]
)
images_df["wrong_to_different_wrong"] = (
    ~images_df["png_correct"]
    & ~images_df["jpg_correct"]
    & images_df["disagreement"]
)
images_df.head()


In [ ]:
jpg_accuracy = images_df["jpg_correct"].mean()
png_accuracy = images_df["png_correct"].mean()
jpg_cross_entropy_loss = images_df["jpg_ce_loss"].mean()
png_cross_entropy_loss = images_df["png_ce_loss"].mean()    
jpg_brier_loss = images_df["jpg_brier_loss"].mean()
png_brier_loss = images_df["png_brier_loss"].mean()

transition_columns = [
    "correct_to_wrong",
    "wrong_to_correct",
    "wrong_to_different_wrong",
]
transition_counts = images_df[transition_columns].sum()
disagreement_count = int(images_df["disagreement"].sum())

assert disagreement_count == int(transition_counts.sum())

summary = pd.Series({
    "source_pairs": len(images_df),
    "jpg_accuracy": jpg_accuracy,
    "png_accuracy": png_accuracy,
    "disagreement_count": disagreement_count,
    "disagreement_rate": images_df["disagreement"].mean(),
    **transition_counts.to_dict(),
    "jpg_cross_entropy_loss": jpg_cross_entropy_loss,
    "png_cross_entropy_loss": png_cross_entropy_loss,
    "jpg_brier_loss": jpg_brier_loss,
    "png_brier_loss": png_brier_loss,
})
summary


In [ ]:
images_df.loc[
    ~images_df["jpg_correct"] & ~images_df["png_correct"],
    ["index", "label", "png_pred", "jpg_pred", "disagreement"],
]


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import numpy as np

# Validate scores before writing results.
score_columns = [
    "jpg_ce_loss", "png_ce_loss",
    "jpg_brier_loss", "png_brier_loss",
]
assert np.isfinite(images_df[score_columns].to_numpy()).all()
assert len(images_df) == sample_count
assert images_df["index"].is_unique

checkpoint_sha256 = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()

created_at = datetime.now(timezone.utc)
run_id = created_at.strftime("%Y%m%dT%H%M%S%fZ")
output_dir = Path("../results") / run_id
output_dir.mkdir(parents=True, exist_ok=False)

# Preserve integer types for counts in JSON.
count_fields = {
    "source_pairs", "disagreement_count",
    "correct_to_wrong", "wrong_to_correct",
    "wrong_to_different_wrong",
}
metrics = {
    name: int(value) if name in count_fields else float(value)
    for name, value in summary.items()
}

report = {
    "created_at_utc": created_at.isoformat(),
    "checkpoint": {
        "path": str(checkpoint_path),
        "sha256": checkpoint_sha256,
        "architecture": loaded_checkpoint["architecture"],
    },
    "dataset": {
        "name": "MNIST",
        "split": "test",
        "sample_count": sample_count,
        "source_id": "original index in MNIST test split",
    },
    "encoding": encoding_metadata,
    "preprocessing": saved_preprocessing,
    "runtime": {
        "torch_version": str(torch.__version__),
        "device": str(device),
    },
    "metrics": metrics,
}

images_df.to_csv(output_dir / "per_image_results.csv", index=False)
(output_dir / "summary.json").write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
print(f"Saved evaluation to {output_dir.resolve()}")


In [ ]:
device

In [ ]:
## test
from src import inference
import importlib
importlib.reload(inference)

label = test_data.targets[0].item()
jpg_path = os.path.join(path, f"0_label_7.jpg") 
jpg = read_image(jpg_path)

inference_inst = inference.MnistInference(
    checkpoint_path=checkpoint_path, device=device
)
prediction = inference_inst.predict(jpg)
print(f"Prediction for {jpg_path}: {prediction}")


In [ ]:
prediction['input_logits']

In [ ]:
images_df.iloc[0]

In [ ]:
torch.testing.assert_close(prediction['input_logits'], images_df.loc[0, 'jpg_logits'], rtol=1e-5, atol=1e-5)

In [ ]:
pwd

In [ ]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd().parent  # When the notebook runs from notebooks/

result = subprocess.run(
    [sys.executable, "-m", "src.cli", "data/test/1_label_2.jpg"],
    cwd=project_root,
    capture_output=True,
    text=True,
)

In [ ]:
result.returncode


In [ ]:
result.stdout, result.stderr